# Conflict Risk Forecasting from GDELT — BigQuery Pipeline

A reproducible notebook that:
1. Pulls **GDELT Events** (global socio-political events) and **GDELT TV** (TV-caption mentions) from Google BigQuery public datasets.
2. Builds five visualisations (time series, station × time heat map, actor word cloud, CAMEO event-type bar, geographic scatter).
3. Trains a **SARIMAX** time-series model that forecasts daily conflict-event volume using the TV signal as an exogenous regressor.

The default example targets **Israel, Oct 2023 → Oct 2024**, but you can repoint it at any country / period in one cell.

> **Bring your own:** GCP project + BigQuery access. See `README.md` for the one-time setup.


## Install dependencies (run once per environment)

The cell below uses the %pip magic so it always installs into the kernel actually running this notebook. Skip it if you already ran pip install -r requirements.txt in this environment.


In [ ]:
# One-time install of required packages. Safe to re-run; pip skips already-installed deps.
# If you already ran pip install -r requirements.txt in this env, you can skip this cell.
%pip install -q google-cloud-bigquery google-auth db-dtypes pandas-gbq pyarrow pandas numpy matplotlib seaborn wordcloud scikit-learn statsmodels

## 0. Configure — the only cell most users need to touch

Set your own:
- **GCP_PROJECT** — your Google Cloud project ID (or number). This is what BigQuery bills.
- **COUNTRY_CODE / NAME** — FIPS 2-letter code of the country to analyse.
- **TV_KEYWORDS** — words to count in TV captions (lowercase).
- **Date windows** — the analysis range.

You can either edit the values below, or export `GCP_PROJECT` as an environment variable before launching Jupyter.


In [ ]:
import os

# --- BigQuery ---
GCP_PROJECT = os.environ.get("GCP_PROJECT", "YOUR_GCP_PROJECT_ID")   # <-- change me or set env var

# --- Analysis target ---
COUNTRY_CODE = "IS"              # FIPS 2-letter code: IS=Israel, EG=Egypt, US=USA, RS=Russia, UP=Ukraine...
COUNTRY_NAME = "Israel"          # human-readable, used in chart titles

TV_KEYWORDS  = ["israel", "israeli", "gaza", "jerusalem", "netanyahu"]   # lowercase

EVENTS_FROM, EVENTS_TO = "2023-10-10", "2024-10-10"   # GDELT Events window
TV_FROM,     TV_TO     = "2023-10-10", "2024-10-10"   # GDELT TV window (data stream ended 2024-10)

# --- Output directory for saved PNGs ---
OUT_DIR = "."

print(f"Target: {COUNTRY_NAME} (FIPS={COUNTRY_CODE})")
print(f"Events: {EVENTS_FROM} -> {EVENTS_TO}   |   TV: {TV_FROM} -> {TV_TO}")
print(f"TV keywords: {TV_KEYWORDS}")


## 1. One-time setup (do this once per machine)

1. **Create or pick a GCP project** at [console.cloud.google.com](https://console.cloud.google.com/). Note its project ID/number.
2. **Enable BigQuery API** for that project.
3. **Install dependencies** (see `requirements.txt`):
   ```
   pip install -r requirements.txt
   ```
4. **Authenticate.** Pick one of:

   | Method | When to use | How |
   |--------|-------------|-----|
   | Application Default Credentials (ADC) | Local dev / personal notebooks | Install [gcloud CLI](https://cloud.google.com/sdk/docs/install), then `gcloud auth application-default login` and `gcloud auth application-default set-quota-project YOUR_PROJECT_ID` |
   | Short-lived access token | Quick try, no install | `gcloud auth print-access-token` → set `BQ_TOKEN` env var (see cell 2) |
   | Service Account JSON | Servers / CI / shared workloads | Create the SA in the console with `BigQuery User` role, download the JSON, set `GOOGLE_APPLICATION_CREDENTIALS=/path/to/key.json` |

Once any of those is in place, the next cell should print `BigQuery client OK`.


## 2. Connection check

The cell below:
- Validates that `GCP_PROJECT` has been customised (fails fast if not).
- Picks credentials in this order: explicit access token in `BQ_TOKEN` → ADC / service-account env.
- Tests with a free `dry_run` query.


In [ ]:
import os, getpass
from google.cloud import bigquery
import pandas as pd

# =========================================================================
#  Authentication
#  -------------------------------------------------------------------------
#  This cell ALWAYS asks YOU to paste your own credentials.
#  It deliberately ignores any BQ_TOKEN / GOOGLE_APPLICATION_CREDENTIALS
#  already set on this machine, so the notebook is safe to run publicly
#  without inheriting someone else's project or token.
# =========================================================================

# 1) Wipe any ambient credentials so we cannot accidentally use them.
for _v in ("BQ_TOKEN", "GOOGLE_APPLICATION_CREDENTIALS"):
    os.environ.pop(_v, None)

print("=" * 74)
print(" Google BigQuery authentication  -  paste your OWN credentials below")
print("=" * 74)
print()
print(" Step 1.  Install the gcloud CLI (one-time):")
print("            https://cloud.google.com/sdk/docs/install")
print()
print(" Step 2.  Open a NEW terminal on YOUR computer (PowerShell / cmd /")
print("          bash / zsh - any shell). Run these two commands:")
print()
print("            gcloud auth login")
print("            gcloud auth print-access-token")
print()
print(" Step 3.  Copy the long string the second command prints.")
print("          (It starts with 'ya29.' and is ~200+ chars long.)")
print("          You will paste it below - it will not be echoed.")
print()
print(" Step 4.  Have your GCP project ID handy as well")
print("          (Cloud Console -> project selector at the top-left,")
print("           looks like 'my-project-id-12345').")
print()
print("-" * 74)

# 2) Project ID - prompted every run, NEVER inherited from the environment.
GCP_PROJECT = input("  Your GCP project ID: ").strip()
if not GCP_PROJECT:
    raise RuntimeError("No GCP project ID entered. Aborting.")

# 3) Access token - masked input, mandatory, NEVER inherited.
_bq_token = getpass.getpass("  Paste your BQ access token (hidden input): ").strip()
if not _bq_token:
    raise RuntimeError("No access token entered. Aborting.")

# 4) Build the BigQuery client. The token lives only inside the client object;
#    nothing is written to os.environ, and we delete the local variable below.
from google.oauth2.credentials import Credentials
bq = bigquery.Client(project=GCP_PROJECT, credentials=Credentials(token=_bq_token))

# 5) Scrub the raw token from local scope as soon as the client owns it.
del _bq_token

print()
print(f"  OK - authenticated to project '{bq.project}' with a short-lived token.")
print("  Tokens last about 1 hour. Re-run this cell to refresh when it expires.")


def dry_run_mb(sql: str) -> float:
    """Return BigQuery scan estimate in MB without executing (free, ~ms latency)."""
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    return bq.query(sql, job_config=cfg).total_bytes_processed / 1024 / 1024


## 3. Datasets

[GDELT](https://www.gdeltproject.org/) is an open project that codes worldwide news into machine-readable events. We use three of its BigQuery tables:

| Table | What it has |
|-------|-------------|
| `gdelt-bq.gdeltv2.events_partitioned` | Every coded event since 1920: who, what, where, when, severity (Goldstein), tone (AvgTone) |
| `gdelt-bq.gdeltv2.iatv_1gramsv2` | Per-day-per-station-per-show word counts in TV captions (Internet Archive). Stopped updating Oct 2024. |
| `gdelt-bq.gdeltv2.gkg_partitioned` | Global Knowledge Graph 2.0: one row per news document with themes, entities, and a 7-part tone vector. Used in section 13. |

The core pipeline (sections 6–10) runs on **Events + TV**, which is enough for a forecasting baseline. **GKG 2.0** (section 13) adds a richer, document-level sentiment / narrative signal; it is larger, so we scan it over a small default window and dry-run first. See section 11 for a primer on all three.

**Cost discipline.** All three tables are time-partitioned. *Always* filter on `_PARTITIONTIME` (Events/GKG) or `TIMESTAMP` (TV) — without that, one query can scan hundreds of GB. Google's free tier is 1 TB scanned per month.

## 4. What date ranges are actually available?

A free metadata query against `INFORMATION_SCHEMA.PARTITIONS` shows the min/max partition for each table. Run once when you start a new analysis to know your bounds.


In [ ]:
ranges_sql = """
SELECT 'events_partitioned' AS table_name,
       MIN(PARSE_DATE('%Y%m%d', partition_id)) AS min_date,
       MAX(PARSE_DATE('%Y%m%d', partition_id)) AS max_date,
       COUNT(*) AS partitions
FROM `gdelt-bq.gdeltv2.INFORMATION_SCHEMA.PARTITIONS`
WHERE table_name = 'events_partitioned' AND partition_id != '__NULL__'
UNION ALL
SELECT 'iatv_1gramsv2',
       MIN(PARSE_DATE('%Y%m%d', partition_id)),
       MAX(PARSE_DATE('%Y%m%d', partition_id)),
       COUNT(*)
FROM `gdelt-bq.gdeltv2.INFORMATION_SCHEMA.PARTITIONS`
WHERE table_name = 'iatv_1gramsv2' AND partition_id != '__NULL__'
"""
ranges_df = bq.query(ranges_sql).result().to_dataframe()
ranges_df


## 5. FIPS country codes (quick reference)

GDELT uses FIPS, not ISO. Examples:

| Country | FIPS | Country | FIPS |
|---------|------|---------|------|
| United States | `US` | Russia | `RS` |
| Israel | `IS` | Ukraine | `UP` |
| Egypt | `EG` | Iran | `IR` |
| Saudi Arabia | `SA` | China | `CH` |
| United Kingdom | `UK` | Germany | `GM` |
| France | `FR` | India | `IN` |

For the full list see the [FIPS 10-4 country codes table](https://en.wikipedia.org/wiki/List_of_FIPS_country_codes).


## 6. Fetch GDELT Events

We pull every event whose action took place in the target country in the window. Key columns:

- `SQLDATE` (int YYYYMMDD) — event date
- `EventCode`, `EventBaseCode`, `EventRootCode` — CAMEO event-type codes
- `Actor1Name`, `Actor2Name` — parties
- `AvgTone` (-100..+100) — sentiment of articles reporting the event
- `GoldsteinScale` (-10..+10) — event severity (negative = destabilising)
- `NumMentions`, `NumSources` — coverage volume
- `ActionGeo_*` — location
- `SOURCEURL` — link to a source article

CAMEO root codes worth knowing: `04` Consult, `14` Protest, `18` Assault, `19` Fight, `20` Unconventional.


In [ ]:
import time

events_sql = f"""
SELECT
  PARSE_DATE('%Y%m%d', CAST(SQLDATE AS STRING)) AS date,
  SQLDATE AS DAY,
  EventCode, EventBaseCode, EventRootCode,
  Actor1Name, Actor2Name,
  CAST(AvgTone        AS FLOAT64) AS AvgTone,
  CAST(GoldsteinScale AS FLOAT64) AS GoldsteinScale,
  CAST(NumMentions    AS INT64)   AS NumMentions,
  CAST(NumSources     AS INT64)   AS NumSources,
  ActionGeo_FullName,
  CAST(ActionGeo_Lat  AS FLOAT64) AS ActionGeo_Lat,
  CAST(ActionGeo_Long AS FLOAT64) AS ActionGeo_Long,
  SOURCEURL
FROM `gdelt-bq.gdeltv2.events_partitioned`
WHERE _PARTITIONTIME >= TIMESTAMP("{EVENTS_FROM}")
  AND _PARTITIONTIME <  TIMESTAMP("{EVENTS_TO}")
  AND ActionGeo_CountryCode = "{COUNTRY_CODE}"
  AND NumMentions > 0
"""

print(f"Dry-run scan estimate: {dry_run_mb(events_sql):,.1f} MB")
print("Running query (BigQuery Storage API for fast download)...")
_t0 = time.time()

# Try BQ Storage API first (fast); silently fall back to REST if the package is missing.
try:
    from google.cloud import bigquery_storage  # noqa: F401
    events_df = bq.query(events_sql).result().to_dataframe(
        create_bqstorage_client=True,
        progress_bar_type=None,
    )
    _mode = "BQ Storage API"
except Exception as _e:
    print(f"  (Storage API unavailable: {type(_e).__name__}: {_e}. Falling back to REST.)")
    events_df = bq.query(events_sql).result().to_dataframe(progress_bar_type=None)
    _mode = "REST"

events_df["date"] = pd.to_datetime(events_df["date"])
print(f"Got {len(events_df):,} events from {events_df['date'].min().date()} to {events_df['date'].max().date()}")
print(f"Download mode: {_mode}  |  elapsed: {time.time()-_t0:,.1f}s")
events_df.head()


## 7. Fetch GDELT TV captions

`iatv_1gramsv2` holds (date, hour, station, show, ngram, count) rows. We aggregate to daily-per-station-per-show-per-keyword to keep the result manageable.

Top stations available: `BBCNEWS`, `ALJAZ` (Al Jazeera), `CNN`, `MSNBC`, `FOXNEWS`, `RT`, `DW`, `CSPAN`, `KQED`.


In [ ]:
keywords_csv = ", ".join(f'"{k}"' for k in TV_KEYWORDS)

tv_sql = f"""
SELECT
  DATE(TIMESTAMP) AS date,
  STATION,
  NGRAM,
  SHOW,
  SUM(COUNT) AS mentions
FROM `gdelt-bq.gdeltv2.iatv_1gramsv2`
WHERE TIMESTAMP >= TIMESTAMP("{TV_FROM}")
  AND TIMESTAMP <  TIMESTAMP("{TV_TO}")
  AND LOWER(NGRAM) IN ({keywords_csv})
GROUP BY date, STATION, NGRAM, SHOW
"""

print(f"Dry-run scan estimate: {dry_run_mb(tv_sql):,.1f} MB")
print("Running query...")
tv_df = bq.query(tv_sql).result().to_dataframe()
tv_df["date"]     = pd.to_datetime(tv_df["date"])
tv_df["mentions"] = tv_df["mentions"].astype("int64")
print(f"Got {len(tv_df):,} (date,station,show,ngram) rows from {tv_df['date'].min().date()} to {tv_df['date'].max().date()}")
tv_df.head()


## 8. Sanity-check summary

Numbers that should look sensible. Zero rows, astronomical sums, or all-positive AvgTone for a conflict zone = sign that the country code / date range / keyword set is wrong.


In [ ]:
print(f"=== {COUNTRY_NAME} — GDELT summary ===\n")

print(f"EVENTS: {len(events_df):,} rows")
print(f"  Daily volume:  mean={events_df.groupby('date').size().mean():.1f}  max={events_df.groupby('date').size().max():,}")
print(f"  AvgTone mean:  {events_df['AvgTone'].mean():.2f}  (negative = hostile coverage)")
print(f"  Goldstein:     mean={events_df['GoldsteinScale'].mean():.2f}  (negative = destabilising)")
print("  Top event roots:")
print(events_df.groupby('EventRootCode').size().sort_values(ascending=False).head(5).to_string())

print(f"\nTV: {len(tv_df):,} rows")
print("  Top stations by total mentions:")
print(tv_df.groupby('STATION')['mentions'].sum().sort_values(ascending=False).head(10).to_string())
print("\n  Mentions per keyword:")
print(tv_df.groupby('NGRAM')['mentions'].sum().sort_values(ascending=False).to_string())


---

## 9. Visualisations

Five charts that cover the data's four key axes: intensity over time, TV coverage strategy, key actors, event mix, and geography. Every figure is also saved as a PNG in `OUT_DIR` for downstream reports.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
print("viz environment ready")


### 9.1 Daily time series — volume, sentiment, severity

Three series on a shared x-axis. The faint line is the raw daily value; the bold line is a 7-day rolling mean that cuts the daily-cycle noise.


In [ ]:
daily = (events_df
         .groupby("date")
         .agg(n_events=("EventCode", "size"),
              avg_tone=("AvgTone", "mean"),
              goldstein=("GoldsteinScale", "mean"))
         .sort_index())
roll = daily.rolling(7, min_periods=1).mean()

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
axes[0].plot(daily.index, daily["n_events"], color="steelblue", alpha=0.35, lw=0.7, label="daily")
axes[0].plot(roll.index,  roll["n_events"],  color="steelblue", lw=2, label="7d rolling")
axes[0].set_ylabel("events / day"); axes[0].legend(loc="upper right")
axes[0].set_title(f"{COUNTRY_NAME}: daily conflict intensity")

axes[1].plot(daily.index, daily["avg_tone"], color="darkorange", alpha=0.35, lw=0.7)
axes[1].plot(roll.index,  roll["avg_tone"],  color="darkorange", lw=2)
axes[1].axhline(0, color="black", lw=0.6, ls="--")
axes[1].set_ylabel("AvgTone\n(- hostile, + positive)")

axes[2].plot(daily.index, daily["goldstein"], color="firebrick", alpha=0.35, lw=0.7)
axes[2].plot(roll.index,  roll["goldstein"],  color="firebrick", lw=2)
axes[2].axhline(0, color="black", lw=0.6, ls="--")
axes[2].set_ylabel("Goldstein\n(- destabilising)")
axes[2].set_xlabel("date")

fig.tight_layout()
fig.savefig(f"{OUT_DIR}/viz_01_timeseries.png", dpi=140, bbox_inches="tight")
plt.show()
print("saved: viz_01_timeseries.png")


### 9.2 Heat map — TV stations × time

Weekly sum of caption mentions for the top 12 stations. Rows = stations, columns = weeks. Log colour scale so a few dominant stations (BBC, Al Jazeera…) don't wash everyone else out.


In [ ]:
tv_top = tv_df.groupby("STATION")["mentions"].sum().nlargest(12).index
tv_h = (tv_df[tv_df["STATION"].isin(tv_top)]
        .assign(week=lambda d: d["date"].dt.to_period("W").dt.start_time)
        .groupby(["STATION", "week"])["mentions"].sum()
        .unstack(fill_value=0)
        .reindex(index=tv_top))

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(np.log1p(tv_h.astype(float)),
            cmap="YlOrRd", linewidths=0,
            cbar_kws={"label": "log(1 + mentions)"},
            ax=ax)
ax.set_title(f"TV mentions of {COUNTRY_NAME} keywords by station x week (log scale)")
ax.set_xlabel("week"); ax.set_ylabel("station")

step = max(1, tv_h.shape[1] // 20)
ax.set_xticks(np.arange(0, tv_h.shape[1], step) + 0.5)
ax.set_xticklabels([d.strftime("%Y-%m-%d") for d in tv_h.columns[::step]], rotation=45, ha="right")

fig.tight_layout()
fig.savefig(f"{OUT_DIR}/viz_02_tv_heatmap.png", dpi=140, bbox_inches="tight")
plt.show()
print("saved: viz_02_tv_heatmap.png")


### 9.3 Word cloud — key actors

`Actor1Name` + `Actor2Name` weighted by `NumMentions` (so the most-reported actors are biggest). Generic / vague tokens (`GOVERNMENT`, `UNKNOWN`, …) are filtered out.


In [ ]:
from wordcloud import WordCloud
from collections import Counter

STOP = {"UNKNOWN", "GOVERNMENT", "OFFICIAL", "REBEL", "CIVILIAN", "MILITANT",
        "NONE", "", "INTERNATIONAL", "PARLIAMENT", "MINISTER", "MILITARY",
        "WORLD", "PEOPLE", "FOREIGN", "WOMAN", "MAN", "REGION", "GROUP"}

cnt = Counter()
for col in ("Actor1Name", "Actor2Name"):
    s = events_df[[col, "NumMentions"]].dropna()
    s = s[~s[col].isin(STOP)]
    for name, n in zip(s[col], s["NumMentions"].astype(int)):
        cnt[name] += n

wc = WordCloud(width=1400, height=700, background_color="white",
               colormap="viridis", max_words=120,
               prefer_horizontal=0.9).generate_from_frequencies(dict(cnt))

fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(wc, interpolation="bilinear"); ax.axis("off")
ax.set_title(f"Top actors in {COUNTRY_NAME} events (size = total NumMentions)")
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/viz_03_wordcloud.png", dpi=140, bbox_inches="tight")
plt.show()
print("saved: viz_03_wordcloud.png — top 10:", cnt.most_common(10))


### 9.4 CAMEO event-type distribution

Bar chart of all CAMEO root codes, red = hostile/coercive (codes 13-20), blue = cooperative/communicative (01-12).


In [ ]:
CAMEO_LABELS = {
    "01": "Public statement",  "02": "Appeal",         "03": "Intent to cooperate",
    "04": "Consult",            "05": "Diplomatic coop", "06": "Material coop",
    "07": "Provide aid",        "08": "Yield",           "09": "Investigate",
    "10": "Demand",             "11": "Disapprove",      "12": "Reject",
    "13": "Threaten",           "14": "Protest",         "15": "Force posture",
    "16": "Reduce relations",   "17": "Coerce",          "18": "Assault",
    "19": "Fight",              "20": "Unconventional",
}
roots = (events_df["EventRootCode"].astype(str).str.zfill(2).value_counts()
         .rename_axis("root").reset_index(name="count"))
roots["label"]   = roots["root"].map(CAMEO_LABELS).fillna("other")
roots["hostile"] = roots["root"].isin(["13","14","15","16","17","18","19","20"])
roots = roots.sort_values("count", ascending=True)

fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(roots["label"] + " (" + roots["root"] + ")",
        roots["count"],
        color=roots["hostile"].map({True: "#c0392b", False: "#2980b9"}))
ax.set_xlabel("event count")
ax.set_title(f"CAMEO root-code distribution — {COUNTRY_NAME} (red = hostile / coercive)")
for i, (cnt_v, lbl) in enumerate(zip(roots["count"], roots["label"])):
    ax.text(cnt_v, i, f"  {cnt_v:,}", va="center", fontsize=8)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/viz_04_cameo_dist.png", dpi=140, bbox_inches="tight")
plt.show()
print("saved: viz_04_cameo_dist.png")


### 9.5 Geographic scatter

One point per unique (lat, lon). Size = log(NumMentions), colour = AvgTone (red = hostile, green = positive). The default lat/lon clip is set for the Israel region; **update `LAT_MIN/MAX` and `LON_MIN/MAX` below for your country**.


In [ ]:
# Geographic clip — change for your country
LAT_MIN, LAT_MAX = 29.0, 34.5
LON_MIN, LON_MAX = 33.5, 37.0

geo = events_df.dropna(subset=["ActionGeo_Lat","ActionGeo_Long"]).copy()
for col in ("ActionGeo_Lat","ActionGeo_Long","AvgTone","GoldsteinScale","NumMentions"):
    geo[col] = geo[col].astype(float)
geo = geo[(geo["ActionGeo_Lat"].between(LAT_MIN, LAT_MAX)) &
          (geo["ActionGeo_Long"].between(LON_MIN, LON_MAX))]
agg = (geo.groupby(["ActionGeo_Lat","ActionGeo_Long","ActionGeo_FullName"])
        .agg(n=("EventCode","size"),
             tone=("AvgTone","mean"),
             mentions=("NumMentions","sum"))
        .reset_index())
agg[["ActionGeo_Lat","ActionGeo_Long","tone","mentions"]] = (
    agg[["ActionGeo_Lat","ActionGeo_Long","tone","mentions"]].astype(float))

fig, ax = plt.subplots(figsize=(10, 11))
sc = ax.scatter(agg["ActionGeo_Long"].values, agg["ActionGeo_Lat"].values,
                c=agg["tone"].values, cmap="RdYlGn", vmin=-10, vmax=10,
                s=np.log1p(agg["mentions"])*4, alpha=0.55,
                edgecolor="black", linewidth=0.2)
plt.colorbar(sc, ax=ax, label="AvgTone")
ax.set(xlabel="Longitude", ylabel="Latitude",
       title=f"Event geography — {COUNTRY_NAME} (size = log mentions, colour = tone)")
top = agg.nlargest(8, "n")
for _, r in top.iterrows():
    ax.annotate(str(r["ActionGeo_FullName"]).split(",")[0],
                (r["ActionGeo_Long"], r["ActionGeo_Lat"]),
                fontsize=7, xytext=(4,3), textcoords="offset points")
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/viz_05_geo.png", dpi=140, bbox_inches="tight")
plt.show()
print("saved: viz_05_geo.png  (", len(agg), "unique points)")


---

## 10. ML model — **SARIMAX** with exogenous TV signal

### Why SARIMAX?

| Requirement | Why SARIMAX fits |
|-------------|-----------------|
| Daily series with weekly seasonality | The `(P,D,Q,s=7)` seasonal block captures weekday/weekend reporting rhythms |
| Useful exogenous covariate (TV mentions) | The **X** in SARIMA**X** = `eXogenous regressors`, built for this |
| Need interpretable output | `summary()` returns coefficients, p-values, AIC — not a black box |
| Small sample (~365 days) | Right zone for classical state-space; LSTM would overfit, Prophet is heavier to install |
| Confidence intervals on the forecast | `get_forecast(...).conf_int()` built-in |

Alternatives we considered:
- **Prophet** — needs cmdstanpy / Stan; heavy to install on Windows.
- **LSTM / Transformer** — data is too small; high overfit risk.
- **XGBoost + lag features** — strong forecast accuracy but no time-series interpretation, no built-in CI.

### Problem formulation

- **Target (y):** daily count of violence events (CAMEO root in {18 Assault, 19 Fight, 20 Unconventional}).
- **Exogenous (X):** log-1-plus of total daily TV mentions across all keywords.
- **Train / test:** last 60 days held out as out-of-sample.
- **Order:** SARIMAX(1,1,1)(1,1,1,7) — a sensible default for daily news series.


In [ ]:
# --- 10.1  Build daily target + exogenous regressor ---
ev = events_df.copy()
ev["is_violence"] = ev["EventRootCode"].astype(str).str.zfill(2).isin(["18","19","20"])

daily_tgt = (ev.groupby("date")
               .agg(violence=("is_violence","sum"),
                    n_events=("EventCode","size"),
                    avg_tone=("AvgTone","mean"))
               .sort_index())

tv_daily = tv_df.groupby("date")["mentions"].sum().astype(float).rename("tv_mentions")

ts = (daily_tgt.join(tv_daily, how="left")
              .asfreq("D")
              .ffill())
ts["tv_log"] = np.log1p(ts["tv_mentions"].fillna(0))

print("merged daily series:", ts.shape, "| range:", ts.index.min().date(), "->", ts.index.max().date())
print(ts.tail(5))


In [ ]:
# --- 10.2  Train / test split + fit SARIMAX ---
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings("ignore")

TEST_DAYS = 60
y      = ts["violence"].astype(float)
X_exog = ts[["tv_log"]].astype(float)

y_train, y_test = y.iloc[:-TEST_DAYS], y.iloc[-TEST_DAYS:]
X_train, X_test = X_exog.iloc[:-TEST_DAYS], X_exog.iloc[-TEST_DAYS:]

print(f"train: {len(y_train)} days  ({y_train.index.min().date()} -> {y_train.index.max().date()})")
print(f"test:  {len(y_test)} days  ({y_test.index.min().date()} -> {y_test.index.max().date()})")

model = SARIMAX(y_train, exog=X_train,
                order=(1,1,1), seasonal_order=(1,1,1,7),
                enforce_stationarity=False, enforce_invertibility=False)
fit = model.fit(disp=False, maxiter=200)
print(fit.summary().tables[0])
print(fit.summary().tables[1])


In [ ]:
# --- 10.3  Forecast + accuracy ---
from sklearn.metrics import mean_absolute_error, mean_squared_error

fc       = fit.get_forecast(steps=TEST_DAYS, exog=X_test)
y_pred   = fc.predicted_mean
ci       = fc.conf_int(alpha=0.05)
ci.columns = ["lo","hi"]

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mask = y_test > 0
mape = (np.abs(y_test[mask]-y_pred[mask])/y_test[mask]).mean()*100
naive_pred = np.repeat(y_train.iloc[-7:].mean(), TEST_DAYS)
naive_mae  = mean_absolute_error(y_test, naive_pred)
skill      = 1 - mae/naive_mae

print(f"SARIMAX  -  MAE={mae:.1f}   RMSE={rmse:.1f}   MAPE={mape:.1f}%")
print(f"7d-mean naive baseline MAE={naive_mae:.1f}")
print(f"Skill score vs naive   = {skill:+.1%}  (positive = better than naive)")

fig, ax = plt.subplots(figsize=(14,5.5))
ax.plot(y_train.index[-180:], y_train.values[-180:], color="steelblue", lw=0.9, label="train (last 180d)")
ax.plot(y_test.index, y_test.values, color="black", lw=1.4, label="actual")
ax.plot(y_test.index, y_pred.values, color="crimson", lw=1.6, label="SARIMAX forecast")
ax.fill_between(y_test.index, ci["lo"], ci["hi"], color="crimson", alpha=0.18, label="95% CI")
ax.set(title=f"{COUNTRY_NAME} violence-events daily forecast - SARIMAX(1,1,1)(1,1,1,7) + TV exog",
       xlabel="date", ylabel="violence-coded events / day")
ax.legend(loc="upper left")
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/viz_06_ml_forecast.png", dpi=140, bbox_inches="tight")
plt.show()
print("saved: viz_06_ml_forecast.png")


### 10.4 Residual diagnostics

A well-specified time-series model leaves residuals that look like white noise: zero mean, no autocorrelation, near-normal distribution. The four-panel diagnostic plot is the classical statsmodels check.


In [ ]:
fig = fit.plot_diagnostics(figsize=(13, 8))
fig.suptitle(f"SARIMAX residual diagnostics  ({COUNTRY_NAME})", y=1.02)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/viz_07_residuals.png", dpi=140, bbox_inches="tight")
plt.show()
print("saved: viz_07_residuals.png")


### What this model unlocks

1. **Nowcast.** Given today's TV-mention burst, estimate today's violence-event volume before the full event reports filter into GDELT.
2. **What-if.** Feed a hypothetical `X` (e.g. "TV coverage doubles next week") to read off the implied event-count distribution.
3. **Drift detection.** Persistently signed residuals = reality has diverged from the model — a structural-change alarm.

### Suggested extensions

- **Multi-country:** swap `COUNTRY_CODE` and re-run; combine into a multivariate VAR for cross-country spillovers.
- **Hyperparameter search** over (p,d,q)(P,D,Q,s) by AIC.
- **Add GKG Themes / V2Tone** as additional regressors for a richer sentiment signal than Events AvgTone alone.
- **Quantile-loss XGBoost** vs SARIMAX for tail-risk forecasting.


---

## 11. Background primer — GDELT 2.0 Events, GKG 2.0 & Google BigQuery

A short, self-contained explanation of the three systems this notebook stands on.
Read this once; it makes every query above and below easier to reason about.

### 11.1 GDELT 2.0 Event Database

The **GDELT Project** ("Global Database of Events, Language, and Tone") monitors
news media worldwide — broadcast, print, and web — in over 100 languages and
codes what it reads into a structured, machine-readable record. The **Event
Database** is the "who did what to whom, where, and when" layer:

- Every 15 minutes GDELT publishes new event rows extracted from the global news feed.
- Each event is coded with **CAMEO** (Conflict and Mediation Event Observations),
  a taxonomy of ~300 action types grouped into 20 *root* codes — from `01` Public
  statement and `04` Consult, through `14` Protest, up to `18` Assault, `19` Fight
  and `20` Unconventional mass violence.
- Two signature numeric scores travel with every event:
  - **GoldsteinScale** (−10 … +10) — a theoretically-derived measure of how much
    the event type destabilises or stabilises a country (negative = destabilising).
  - **AvgTone** (−100 … +100) — the average emotional tone of all news articles
    mentioning the event (negative = hostile / negative coverage).
- Coverage volume is captured by **NumMentions**, **NumSources**, **NumArticles**,
  and each event carries actor names, geographic coordinates (`ActionGeo_*`), and a
  `SOURCEURL` back to a source article.

In BigQuery this is `gdelt-bq.gdeltv2.events_partitioned` — the table this notebook
already queries in section 6.

### 11.2 GDELT 2.0 Global Knowledge Graph (GKG)

Where the Event table answers *"what happened?"*, the **Global Knowledge Graph**
answers *"what is the news talking about, and how?"*. Instead of one row per event,
GKG produces one row per **news document**, annotated with everything GDELT could
extract from it:

- **V2Themes** — thousands of thematic tags (e.g. `TAX_FNCACT`, `ARMEDCONFLICT`,
  `REFUGEES`, `TERROR`) drawn from GDELT's GKG Theme taxonomy + the World Bank /
  WHO ontologies.
- **V2Persons / V2Organizations / V2Locations** — named entities, with locations
  resolved to FIPS country codes and lat/long.
- **V2Tone** — a richer, 7-part emotional fingerprint of the whole document:
  `tone, positive_score, negative_score, polarity, activity_density,
  self_group_density, word_count`.
- **GCAM** — the Global Content Analysis Measures: scores from 2,300+ emotion and
  themic dictionaries (anxiety, anger, optimism, etc.) for cross-lingual sentiment.
- **AllNames, Amounts, Quotations, Dates** — additional extracted structure.

GKG is the natural place to go for a **higher-resolution sentiment / narrative
signal** than the Event table's single `AvgTone` number. In BigQuery it is
`gdelt-bq.gdeltv2.gkg_partitioned` — we put it to work in section 13.

### 11.3 Google BigQuery

**BigQuery** is Google Cloud's serverless, columnar data warehouse. Three facts
explain everything about how we use it here:

1. **You pay for bytes *scanned*, not rows returned.** A `SELECT *` over a huge
   table can scan hundreds of GB even with a `LIMIT 5`, because `LIMIT` trims the
   *output*, not the *scan*. The lever that actually reduces cost is selecting
   fewer columns and **filtering on the partition column**.
2. **GDELT's tables are time-partitioned** (`_PARTITIONTIME` on Events/GKG,
   `TIMESTAMP` on TV). Filtering on the partition prunes whole days of data before
   they are ever read — this is why every query in this notebook starts with a date
   filter, and why we never run un-bounded queries.
3. **Dry runs are free.** `dry_run=True` returns the exact byte estimate without
   executing or billing. The `dry_run_mb()` helper (section 2) prints this before
   every real query so you always know the cost first.

GDELT publishes its data as **public datasets** under the `gdelt-bq` project, so
*you* never store or pay for the data — you only pay (or use the free 1 TB/month
tier) for the compute to scan the slices you ask for, billed to **your own**
`GCP_PROJECT`.

---

## 12. Exploratory data analysis — inspect the tables before you trust them

Before building charts or a model, it pays to look at the raw shape of each table:
its **schema** (what columns exist and their types), a **sample** of real rows, and
basic **distributions / missingness**. The cells below do exactly that. Schema
inspection via `bq.get_table()` is a *free metadata call* (zero bytes scanned);
sample queries are kept cheap with a narrow partition window + dry-run estimate.

### 12.1 Events table — schema (what parameters does this table have?)

In [ ]:
# Free metadata call — bq.get_table() scans 0 bytes. Lists every column + type.
events_tbl = bq.get_table("gdelt-bq.gdeltv2.events_partitioned")
print(f"Table : {events_tbl.full_table_id}")
print(f"Rows  : {events_tbl.num_rows:,}")
print(f"Size  : {events_tbl.num_bytes / 1e9:,.1f} GB")
print(f"Cols  : {len(events_tbl.schema)}\n")

events_schema = pd.DataFrame(
    [(f.name, f.field_type, f.mode, (f.description or "")[:70]) for f in events_tbl.schema],
    columns=["column", "type", "mode", "description"],
)
events_schema

### 12.2 Events table — print a real sample of rows

We sample a **2-day** partition window (not the full year) so the scan stays tiny. Remember: `LIMIT` does **not** reduce the scan — the partition filter does.

In [ ]:
sample_events_sql = f"""
SELECT *
FROM `gdelt-bq.gdeltv2.events_partitioned`
WHERE _PARTITIONTIME >= TIMESTAMP("{EVENTS_FROM}")
  AND _PARTITIONTIME <  TIMESTAMP(DATE_ADD(DATE("{EVENTS_FROM}"), INTERVAL 2 DAY))
  AND ActionGeo_CountryCode = "{COUNTRY_CODE}"
LIMIT 5
"""
print(f"Dry-run scan estimate: {dry_run_mb(sample_events_sql):,.1f} MB")
sample_events = bq.query(sample_events_sql).result().to_dataframe()
print(f"{sample_events.shape[1]} columns returned. First 5 matching rows:")
sample_events

### 12.3 Events — types, summary statistics & missing values

Uses the `events_df` already loaded in section 6 — no extra BigQuery cost.

In [ ]:
print("=== dtypes & non-null counts ===")
events_df.info()

print("\n=== numeric summary ===")
display(events_df[["AvgTone", "GoldsteinScale", "NumMentions", "NumSources"]].describe().round(2))

print("\n=== missing values per column ===")
na = events_df.isna().sum()
display(na[na > 0].rename("n_missing").to_frame().assign(
    pct=lambda d: (100 * d["n_missing"] / len(events_df)).round(1)))

### 12.4 Events — deeper EDA: cadence, mix, sources, correlations

In [ ]:
# (a) Monthly event volume — is coverage stable over the window?
monthly = events_df.set_index("date").resample("MS").size().rename("events")
print("=== events per month ===")
print(monthly.to_string())

# (b) Most active actor pairs
pairs = (events_df.dropna(subset=["Actor1Name", "Actor2Name"])
         .groupby(["Actor1Name", "Actor2Name"]).size()
         .sort_values(ascending=False).head(10))
print("\n=== top actor1 -> actor2 pairs ===")
print(pairs.to_string())

# (c) Where do the source articles come from? (domain of SOURCEURL)
domains = (events_df["SOURCEURL"].dropna()
           .str.extract(r"https?://(?:www\.)?([^/]+)")[0]
           .value_counts().head(10))
print("\n=== top source domains ===")
print(domains.to_string())

# (d) Correlation among the numeric signals
print("\n=== correlation matrix (numeric signals) ===")
display(events_df[["AvgTone", "GoldsteinScale", "NumMentions", "NumSources"]].corr().round(2))

### 12.5 TV table — schema + sample

Same two-step inspection for `iatv_1gramsv2` (the TV-caption 1-gram table).

In [ ]:
tv_tbl = bq.get_table("gdelt-bq.gdeltv2.iatv_1gramsv2")
print(f"Table : {tv_tbl.full_table_id}")
print(f"Rows  : {tv_tbl.num_rows:,}   Size: {tv_tbl.num_bytes / 1e9:,.1f} GB   Cols: {len(tv_tbl.schema)}\n")
display(pd.DataFrame([(f.name, f.field_type, f.mode) for f in tv_tbl.schema],
                     columns=["column", "type", "mode"]))

sample_tv_sql = f"""
SELECT DATE(TIMESTAMP) AS date, STATION, SHOW, NGRAM, COUNT
FROM `gdelt-bq.gdeltv2.iatv_1gramsv2`
WHERE TIMESTAMP >= TIMESTAMP("{TV_FROM}")
  AND TIMESTAMP <  TIMESTAMP(DATE_ADD(DATE("{TV_FROM}"), INTERVAL 1 DAY))
  AND LOWER(NGRAM) IN ({keywords_csv})
LIMIT 5
"""
print(f"\nDry-run scan estimate: {dry_run_mb(sample_tv_sql):,.1f} MB")
display(bq.query(sample_tv_sql).result().to_dataframe())

### 12.6 TV — coverage concentration & keyword mix

Uses the `tv_df` loaded in section 7.

In [ ]:
# Share of total mentions captured by the top-N stations (coverage concentration)
by_station = tv_df.groupby("STATION")["mentions"].sum().sort_values(ascending=False)
share = (by_station.cumsum() / by_station.sum() * 100).round(1)
print("=== cumulative % of mentions by station (top 10) ===")
print(pd.concat([by_station.head(10).rename("mentions"),
                 share.head(10).rename("cum_pct")], axis=1).to_string())

print("\n=== mentions per keyword ===")
print(tv_df.groupby("NGRAM")["mentions"].sum().sort_values(ascending=False).to_string())

---

## 13. Using GKG 2.0 in this project

Section 11.2 introduced the **Global Knowledge Graph**. Here we actually pull from
`gdelt-bq.gdeltv2.gkg_partitioned` and turn it into a usable signal — a
**document-level sentiment series** that is richer than the Event table's `AvgTone`,
plus a **thematic breakdown** of what the coverage is about.

**Why bother?** The Event table's `AvgTone` is averaged over articles *for coded
events only*. GKG's `V2Tone` covers **every news document** that mentions the
country, including framing, analysis and opinion pieces that never become a coded
"event". That makes it a strong candidate for an **extra exogenous regressor** in
the SARIMAX model (section 10).

**Cost note.** GKG is a large table. We default to a **90-day** window below so the
demo stays cheap; every query dry-runs first. Widen `GKG_FROM` / `GKG_TO` only after
you have checked the printed scan estimate.

### 13.1 GKG table — schema

Free metadata call. Note the `V2Themes`, `V2Locations`, `V2Persons`, `V2Tone`, `GCAM` columns described in section 11.2. If a column name in the queries below ever errors, check the exact spelling here first.

In [ ]:
# A small GKG window keeps the demo cheap. Default: first 90 days of the events window.
GKG_FROM = EVENTS_FROM
GKG_TO   = (pd.to_datetime(EVENTS_FROM) + pd.Timedelta(days=90)).strftime("%Y-%m-%d")
print(f"GKG window: {GKG_FROM} -> {GKG_TO}  (edit GKG_FROM/GKG_TO to widen)\n")

gkg_tbl = bq.get_table("gdelt-bq.gdeltv2.gkg_partitioned")
print(f"Table : {gkg_tbl.full_table_id}")
print(f"Rows  : {gkg_tbl.num_rows:,}   Size: {gkg_tbl.num_bytes / 1e9:,.1f} GB   Cols: {len(gkg_tbl.schema)}\n")
display(pd.DataFrame([(f.name, f.field_type) for f in gkg_tbl.schema],
                     columns=["column", "type"]))

### 13.2 GKG — sample documents about the target country

GKG resolves locations to FIPS country codes inside the `V2Locations` field (format `...#<FIPS>#...`), so we filter with a `LIKE "%#IS#%"`-style match. One-day window for the sample.

In [ ]:
sample_gkg_sql = f"""
SELECT
  DATE(_PARTITIONTIME)                       AS date,
  SourceCommonName                           AS source,
  DocumentIdentifier                         AS url,
  SAFE_CAST(SPLIT(V2Tone, ',')[OFFSET(0)] AS FLOAT64) AS doc_tone,
  V2Themes
FROM `gdelt-bq.gdeltv2.gkg_partitioned`
WHERE _PARTITIONTIME >= TIMESTAMP("{GKG_FROM}")
  AND _PARTITIONTIME <  TIMESTAMP(DATE_ADD(DATE("{GKG_FROM}"), INTERVAL 1 DAY))
  AND V2Locations LIKE "%#{COUNTRY_CODE}#%"
LIMIT 5
"""
print(f"Dry-run scan estimate: {dry_run_mb(sample_gkg_sql):,.1f} MB")
display(bq.query(sample_gkg_sql).result().to_dataframe())

### 13.3 GKG — daily document tone (a richer sentiment signal)

In [ ]:
gkg_tone_sql = f"""
SELECT
  DATE(_PARTITIONTIME) AS date,
  COUNT(*)             AS n_docs,
  AVG(SAFE_CAST(SPLIT(V2Tone, ',')[OFFSET(0)] AS FLOAT64)) AS gkg_tone,
  AVG(SAFE_CAST(SPLIT(V2Tone, ',')[OFFSET(1)] AS FLOAT64)) AS gkg_positive,
  AVG(SAFE_CAST(SPLIT(V2Tone, ',')[OFFSET(2)] AS FLOAT64)) AS gkg_negative
FROM `gdelt-bq.gdeltv2.gkg_partitioned`
WHERE _PARTITIONTIME >= TIMESTAMP("{GKG_FROM}")
  AND _PARTITIONTIME <  TIMESTAMP("{GKG_TO}")
  AND V2Locations LIKE "%#{COUNTRY_CODE}#%"
GROUP BY date
ORDER BY date
"""
print(f"Dry-run scan estimate: {dry_run_mb(gkg_tone_sql):,.1f} MB")
print("Running query...")
gkg_daily = bq.query(gkg_tone_sql).result().to_dataframe()
gkg_daily["date"] = pd.to_datetime(gkg_daily["date"])
print(f"Got {len(gkg_daily):,} days  ({gkg_daily['date'].min().date()} -> {gkg_daily['date'].max().date()})")
display(gkg_daily.head())

### 13.4 GKG vs Events — compare the two tone signals

Do the document-level GKG tone and the event-level `AvgTone` tell the same story? Overlay them on the GKG window. Strong agreement = GKG is a trustworthy stand-in / complement; divergences are themselves interesting.

In [ ]:
ev_tone = (events_df.set_index("date")["AvgTone"].resample("D").mean()
           .loc[GKG_FROM:GKG_TO].rename("events_AvgTone"))
gk_tone = gkg_daily.set_index("date")["gkg_tone"].rename("gkg_tone")

cmp = pd.concat([ev_tone, gk_tone], axis=1)
r = cmp.corr().iloc[0, 1]

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(cmp.index, cmp["events_AvgTone"], color="darkorange", lw=1.4, label="Events AvgTone")
ax.plot(cmp.index, cmp["gkg_tone"],       color="purple",     lw=1.4, label="GKG V2Tone")
ax.axhline(0, color="black", lw=0.6, ls="--")
ax.set(title=f"{COUNTRY_NAME}: Events AvgTone vs GKG document tone  (Pearson r = {r:.2f})",
       xlabel="date", ylabel="tone (- hostile, + positive)")
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/viz_08_gkg_tone.png", dpi=140, bbox_inches="tight")
plt.show()
print("saved: viz_08_gkg_tone.png")

### 13.5 GKG — what themes drive the coverage?

The `V2Themes` field is a `;`-separated list of `THEME,charOffset` tokens. We unnest it and rank the most-mentioned themes for the country over the GKG window.

In [ ]:
gkg_themes_sql = f"""
SELECT
  SPLIT(raw, ',')[OFFSET(0)] AS theme,
  COUNT(*)                   AS mentions
FROM `gdelt-bq.gdeltv2.gkg_partitioned`,
     UNNEST(SPLIT(V2Themes, ';')) AS raw
WHERE _PARTITIONTIME >= TIMESTAMP("{GKG_FROM}")
  AND _PARTITIONTIME <  TIMESTAMP("{GKG_TO}")
  AND V2Locations LIKE "%#{COUNTRY_CODE}#%"
  AND raw != ''
GROUP BY theme
ORDER BY mentions DESC
LIMIT 25
"""
print(f"Dry-run scan estimate: {dry_run_mb(gkg_themes_sql):,.1f} MB")
gkg_themes = bq.query(gkg_themes_sql).result().to_dataframe()

fig, ax = plt.subplots(figsize=(11, 8))
order = gkg_themes.sort_values("mentions")
ax.barh(order["theme"], order["mentions"], color="#6c3483")
ax.set(xlabel="document mentions", title=f"Top GKG themes — {COUNTRY_NAME} ({GKG_FROM} to {GKG_TO})")
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/viz_09_gkg_themes.png", dpi=140, bbox_inches="tight")
plt.show()
print("saved: viz_09_gkg_themes.png")
display(gkg_themes.head(15))

### 13.6 Wiring GKG tone into the SARIMAX model

Section 10 used `log(1 + TV mentions)` as the single exogenous regressor. The GKG
daily tone is a natural **second regressor**. To use it, extend the exogenous matrix
`X` built in section 10.1 — for example:

```python
# After section 10.1 builds `ts`, join the GKG tone series:
ts = ts.join(gkg_daily.set_index("date")[["gkg_tone"]], how="left")
ts["gkg_tone"] = ts["gkg_tone"].interpolate().bfill().ffill()

# Then in section 10.2, use BOTH regressors:
X_exog = ts[["tv_log", "gkg_tone"]].astype(float)
```

The rest of the pipeline (train/test split, `SARIMAX(...)`, `get_forecast`) is
unchanged — `statsmodels` accepts a multi-column `exog`. Compare the new skill score
against the TV-only baseline to see whether GKG sentiment adds predictive value.

> **Note:** to use GKG over the *full* model window you must widen `GKG_TO` to match
> `EVENTS_TO` and re-run section 13.3 — check the dry-run estimate first, as a full
> year of GKG scans noticeably more than the 90-day demo.

---

## Limitations & responsible use

- GDELT is a **coding of news**, not ground truth. Reporting bias is real; events outside English/global media are under-counted.
- **Correlation, not causation.** TV mentions and events are co-driven by underlying news cycles.
- **GDELT TV stopped Oct 2024.** For analyses extending past that date, drop the TV regressor or substitute another exogenous signal.
- Forecasts are **probabilistic estimates**, not predictions of individual incidents. Use confidence intervals, not point predictions, for any downstream decision.
